<a href="https://colab.research.google.com/github/SuyaNangabo/plp-da-a1-gikomba-stall-analysis/blob/main/Wanjiku_stall_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [115]:
daily_sales = [
    2300, 1950, 2800, 2100, 3600, 5200, 4100,
    2500, 2200, 3100, 2400, 3900, 5600, 4700
]  # 14 days, KES

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] * 2

# qty = kg in stock, reorder = reorder level (kg), cost/price = KES per kg
stock = {
    "Tomatoes": {"qty": 18, "reorder": 25, "cost": 60, "price": 90},
    "Onions":   {"qty": 40, "reorder": 20, "cost": 70, "price": 100},
    "Sukuma":   {"qty": 9,  "reorder": 15, "cost": 30, "price": 50},
    "Potatoes": {"qty": 55, "reorder": 30, "cost": 50, "price": 75},
    "Avocado":  {"qty": 12, "reorder": 10, "cost": 80, "price": 140},
}

## Sales Summary and Best/Worst Days

**Approach:** I used `sum()`, `len()`, `max()` and `min()` to get the total, average, highest and lowest daily sales. To find *when* the best and worst days happened, I used `list.index()` to get each day's position. The day name comes from the `days` list, and the week from `position // 7 + 1`, since positions 0 to 6 are week 1. This tells Wanjiku her overall level and when her extremes occur.

In [116]:
total_daily_sales=sum(daily_sales)
avg_daily_sales=total_daily_sales/len(daily_sales)
highest_daily_sales=max(daily_sales)
lowest_daily_sales=min(daily_sales)

best_day_position=daily_sales.index(highest_daily_sales)
worst_day_position=daily_sales.index(lowest_daily_sales)

best_day, best_week=days[best_day_position], best_day_position // 7 + 1
worst_day, worst_week=days[worst_day_position], worst_day_position // 7 + 1


print(f"Total daily sales: KES {total_daily_sales:,.2f}")
print(f"Average daily sales:KES {avg_daily_sales:,.2f}")
print(f"Highest daily sales: KES {highest_daily_sales:,.2f}")
print(f"Lowest daily sales: KES {lowest_daily_sales:,.2f}")

print(f"The best day was {best_day} with KES {highest_daily_sales:,.2f} ({best_day}, week {best_week})")
print(f"The worst day was {worst_day} with KES {lowest_daily_sales}({worst_day}, week {worst_week})")




Total daily sales: KES 46,450.00
Average daily sales:KES 3,317.86
Highest daily sales: KES 5,600.00
Lowest daily sales: KES 1,950.00
The best day was Sat with KES 5,600.00 (Sat, week 2)
The worst day was Tue with KES 1950(Tue, week 1)


## Weekly Totals and Growth

**Approach:** I wrote `week_totals(sales)`, which uses slicing (`sales[:7]` and `sales[7:]`) to total each week and returns both values as a tuple. I unpacked them into two variables and calculated growth as (week 2 − week 1) ÷ week 1 × 100. Comparing whole weeks smooths out daily ups and downs, so it answers "is my business growing?" fairly.

In [117]:
def week_totals(sales):
  """Return (week 1 total, week 2 total)"""
  week1=sum(sales[:7])
  week2=sum(sales[7:])
  return (week1, week2)

week1_total, week2_total=week_totals(daily_sales)
growth=(week2_total-week1_total)/week1_total*100

print(f"Week 1 total: KES {week1_total:,.2f}")
print(f"Week 2 total: KES {week2_total:,.2f}")
print(f"Growth: {growth:.1f}%")



Week 1 total: KES 22,050.00
Week 2 total: KES 24,400.00
Growth: 10.7%


## Classifying Strong, Normal and Weak Days

**Approach:** I wrote `classify_day(amount, average)`, which labels a day Strong if it is more than 20% above the average, Weak if more than 20% below, and Normal otherwise. I applied it to all 14 days and counted each label with the dictionary counting pattern (`counts.get(label, 0) + 1`). This shows how often Wanjiku has good and bad days, which an average alone hides.

In [118]:
def Classify_day(amount, average):
  """ Return 'Strong' (>20% above average), 'Normal', or 'Weak' (>20% below average)"""
  if amount > average * 1.2:
    return "Strong"
  elif amount < average * 0.8:
    return "Weak"
  else:
    return "Normal"

counts={}
for amount in daily_sales:
  label=Classify_day(amount, avg_daily_sales)
  counts[label]=counts.get(label, 0) + 1

print(f"{'Day':<5}{'Amount':>10}{'Label':>10}")
for day, amount, label in zip(days, daily_sales, [Classify_day(amount, avg_daily_sales) for amount in daily_sales]):
  print(f"{day:<5}{amount:>10,.2f}{label:>10}")

print()
print("Counts:", counts)

Day      Amount     Label
Mon    2,300.00      Weak
Tue    1,950.00      Weak
Wed    2,800.00    Normal
Thu    2,100.00      Weak
Fri    3,600.00    Normal
Sat    5,200.00    Strong
Sun    4,100.00    Strong
Mon    2,500.00      Weak
Tue    2,200.00      Weak
Wed    3,100.00    Normal
Thu    2,400.00      Weak
Fri    3,900.00    Normal
Sat    5,600.00    Strong
Sun    4,700.00    Strong

Counts: {'Weak': 6, 'Normal': 4, 'Strong': 4}


## Weekdays vs Weekends

**Approach:** I looped through the days and sales together, sorting each sale into a weekday list (Mon to Fri) or a weekend list (Sat, Sun). I compared **averages per day**, not totals, because there are 10 weekdays but only 4 weekend days. The difference is (weekend avg − weekday avg) ÷ weekday avg × 100. This shows how much more the stall earns on weekends.

In [119]:
weekday_sales=[]
weekend_sales=[]

for day, amount in zip(days, daily_sales):
  if day in ["Sat", "Sun"]:
    weekend_sales.append(amount)
  else:
    weekday_sales.append(amount)


weekday_avg=sum(weekday_sales)/len(weekday_sales)
weekend_avg=sum(weekend_sales)/len(weekend_sales)

weekend_lift_percentage=(weekend_avg-weekday_avg) / weekday_avg * 100
weekend_share_percentage = sum(weekend_sales) / total_daily_sales * 100


print(f"Weekday average:(Mon-Fri) KES {weekday_avg:,.2f}")
print(f"Weekend average:(Sat-Sun) KES {weekend_avg:,.2f}")
print(f"Weekends sell: {weekend_lift_percentage:.1f}% more per day than weekdays")
print(f"Weekends are 4 of 14 days but bring in {weekend_share_percentage:.1f}% of all sales")




Weekday average:(Mon-Fri) KES 2,685.00
Weekend average:(Sat-Sun) KES 4,900.00
Weekends sell: 82.5% more per day than weekdays
Weekends are 4 of 14 days but bring in 42.2% of all sales


## Restock List and Gross Margins

**Approach:** I wrote `products_to_restock()`, which loops through the stock dictionary with `.items()` and returns every product with quantity at or below its reorder level (`<=`). I then calculated gross margin as (price − cost) ÷ price × 100 for each product. The restock list shows what to buy now, and the margins show which products earn the most from each shilling of sales.

In [120]:
def products_to_restock(stock_data):
  """ Return a list of products whose quantity is at or below the reorder level"""
  restock_list=[]
  for product, data in stock_data.items ():
    if data["qty"] <= data["reorder"]:
     restock_list.append(product)
  return restock_list

restock_list=products_to_restock(stock)

print("Products to restock:", restock_list)






Products to restock: ['Tomatoes', 'Sukuma']


## Stall Performance Report

**Approach:** I combined every result into one printed report using f-strings. A small `format_kes()` function formats money as KES with thousands separators, and alignment codes (`<`, `>`, `^`) keep the columns tidy. The aim is a one-page summary Wanjiku can read quickly without any coding knowledge.

In [121]:
# Calculate margins first since it hasn't been defined in previous cells
margins = {}
for product, data in stock.items():
    margins[product] = ((data["price"] - data["cost"]) / data["price"]) * 100

# THE ALIGNMENT TOOLS (INSIDE THE{} OF AN F-STRING)
# {text:<26} - left-aligns in 26 characters.
# {text:>24} - right-aligns in 24 characters, which lines up the digits.
# {text:^52} - centres in 52 characters.
line="=" * 52
thin="-" * 52

print(line)
print(f"{'STALL PERFORMANCE REPORT':^52}")
print(f"{'Wanjiku | Gikomba Market | 14-Day Review':^52}")
print(line)

print("SALES SUMMARY")
print(f" {'Total sales':<26}{f'KES {total_daily_sales:,.2f}':>24}")
print(f" {'Average daily sales':<26}{f'KES {avg_daily_sales:,.2f}':>24}")
print(f" {'Best day':<26}{f'KES {highest_daily_sales:,.2f} ({best_day}, Week {best_week})':>24}")
print(f" {'Worst day':<26}{f'KES {lowest_daily_sales:,.2f} ({worst_day}, Week {worst_week})':>24}")
print(thin)

print("GROWTH")
print(f" {'Week 1 total':<26}{f'KES {week1_total:,.2f}':>24}")
print(f" {'Week 2 total':<26}{f'KES {week2_total:,.2f}':>24}")
print(f" {'Week-on-week growth':<26}{f'{growth:.1f}%':>24}")
print(thin)

print("DAY PERFORMANCE (vs average)")
for label in ("Strong", "Normal", "Weak"):
  print(f" {label + ' days':<26}{counts.get(label, 0):>24}")
print(thin)

print("WEEKDAYS VS WEEKENDS")
print(f" {'Weekday average':<26}{f'KES {weekday_avg:,.2f}':>24}")
print(f" {'Weekend average':<26}{f'KES {weekend_avg:,.2f}':>24}")
print(f" {'Weekend lift':<26}{f'{weekend_lift_percentage:.1f}%':>24}")
print(thin)

print("STOCK AND MARGINS")
print(f" {'Product':<12}{'In stock':>10}{'Reorder':>10}{'Margin':>10}{'Action':>8}")
for product, data in stock.items():
  action="RESTOCK" if product in restock_list else "OK"
  print(f" {product:<12}{str(data['qty']) + 'kg':>10}{str(data['reorder']) + 'kg':>10}"
        f"{margins[product]:>9.1f}%{action:>8}")
print(line)


              STALL PERFORMANCE REPORT              
      Wanjiku | Gikomba Market | 14-Day Review      
SALES SUMMARY
 Total sales                          KES 46,450.00
 Average daily sales                   KES 3,317.86
 Best day                  KES 5,600.00 (Sat, Week 2)
 Worst day                 KES 1,950.00 (Tue, Week 1)
----------------------------------------------------
GROWTH
 Week 1 total                         KES 22,050.00
 Week 2 total                         KES 24,400.00
 Week-on-week growth                          10.7%
----------------------------------------------------
DAY PERFORMANCE (vs average)
 Strong days                                      4
 Normal days                                      4
 Weak days                                        6
----------------------------------------------------
WEEKDAYS VS WEEKENDS
 Weekday average                       KES 2,685.00
 Weekend average                       KES 4,900.00
 Weekend lift                       

## Business Recommendations

**Approach:** Each recommendation below comes directly from a number in Tasks 1 to 6, so every suggestion is backed by evidence rather than opinion.



## Recommendations for Wanjiku

1. **Keep going:** the stall is growing, with sales up **10.7%** from KES 22,050 in week 1 to KES 24,400 in week 2, so the current way of trading is working.
2. **Restock tomatoes and sukuma before Friday**, because they are below their reorder levels (18 kg vs 25 kg and 9 kg vs 15 kg) and weekends average **KES 4,900 a day, 82.5% more** than weekdays.
3. **Promote avocado, especially on weekends:** it has the highest margin at **42.9%** and only 12 kg in stock (2 kg above its reorder level), so top it up too.
4. **Stop buying onions and potatoes for now:** onions have the lowest margin (**30%**) and 40 kg in stock, double the 20 kg reorder level, and potatoes hold 55 kg against a 30 kg level, so cash is tied up in slow stock.
5. **Lift the weak days:** Monday, Tuesday and Thursday were Weak on **6 of 14 days** (below KES 2,654), so a small offer or bundle on those days could raise them toward the KES 3,318 average.